In [1]:
import pandas as pd

In [3]:



#Taking a sample because dealing with 99 million rows of data at once is computationally heavy

import pyarrow as pa
import pyarrow.parquet as pq

source = pq.ParquetFile("Data/2026-09.parquet")
output = "Data/sample_100k.parquet"

remaining = 100_000

with pq.ParquetWriter(output, source.schema_arrow) as writer:
    for batch in source.iter_batches(batch_size=10_000):
        batch = batch.slice(0, min(batch.num_rows, remaining))

        writer.write_table(pa.Table.from_batches([batch]))
        remaining -= batch.num_rows

        print(f"Remaining: {remaining:,}")

        if remaining == 0:
            break

print("Sample saved!")

Remaining: 90,000
Remaining: 80,000
Remaining: 70,000
Remaining: 60,000
Remaining: 50,000
Remaining: 40,000
Remaining: 30,000
Remaining: 20,000
Remaining: 10,000
Remaining: 0
Sample saved!


In [2]:
df = pd.read_parquet('Data/sample_100k.parquet')

In [3]:
df['is_ai_generated'].sum() #5000 out of 120k are ai 

np.float64(5091.0)

In [18]:
df.dtypes

video_id                                uint64
posted_at                  datetime64[ms, UTC]
country                                    str
language                                   str
type                                       str
duration_ms                             uint32
image_count                              uint8
width                                   uint16
height                                  uint16
caption                                    str
hashtags                                object
hashtag_ids                             object
mentions                                object
on_screen_text                          object
is_ad                                    uint8
branded_content                            str
product_id                              uint64
is_shop_video                            uint8
seller_id                               uint64
is_ai_generated                        float64
not_recommended_for_fyp                  uint8
sound_muted  

In [5]:
print(df['language'].unique())
print(len(df[df['language'] == 'un']))

<ArrowStringArray>
[ 'un',  'en',  'ar',  'km',  'nl',  'ru',  'id',  'es',  'sr',  'de',
 ...
  'tw',  'kn', 'kmr',  'ga', 'nso',  'ti',  'be',  'to',  'hy',  'gd']
Length: 114, dtype: str
58662


In [19]:
dl = df.groupby('language').mean(numeric_only=True)

In [20]:
stats = (df.groupby('language')['is_ai_generated']
           .agg(rate='mean', n='size', labeled='count')
           .sort_values('n', ascending=False))

stats[stats['labeled'] >= 30].sort_values('rate', ascending=False)



,rate,n,labeled
language,,,
zh-Hans,0.223684,93,76
pl,0.179104,81,67
ta,0.169811,59,53
om,0.121212,80,66
ro,0.102564,48,39
nl,0.102564,46,39
it,0.100917,124,109
ne,0.096059,508,406
si,0.093333,174,150


In [21]:
import numpy as np
df['caption'] = df['caption'].str.strip().replace('', np.nan)

In [22]:
#we would like to take the un and fill it using the caption since it has much data about the language of the video 
df['caption'].isna().sum()
df[(df['language'] == 'un') & (df['caption'].isna())]

,video_id,posted_at,country,language,type,duration_ms,image_count,width,height,caption,...,not_recommended_for_fyp,sound_muted,views,likes,comments,shares,saves,downloads,engagement_rate,stats_updated_at
5,7680354140752497934,2026-09-01 00:17:01+00:00,--,un,video,10000,0,0,0,NaN,...,0,0,283,38,1,0,0,0,0.137809,2026-09-17 23:07:58+00:00
6,7680354325939473677,2026-09-01 00:17:42+00:00,--,un,video,15000,0,0,0,NaN,...,0,0,346,43,0,0,0,0,0.124277,2026-09-23 23:59:59+00:00
12,7680363720639253790,2026-09-01 00:54:00+00:00,--,un,video,15046,0,1080,1920,NaN,...,0,0,347,35,1,0,4,0,0.115274,2026-09-30 00:13:02+00:00
14,7680364013917621535,2026-09-01 00:55:10+00:00,--,un,video,13000,0,0,0,NaN,...,0,0,269,3,0,0,0,0,0.011152,2026-09-18 06:46:37+00:00
15,7680366081042648340,2026-09-01 01:03:07+00:00,--,un,video,57234,0,720,1280,NaN,...,0,0,168,21,4,1,2,1,0.166667,2026-09-30 01:12:41+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99984,7682349797663149320,2026-09-06 09:20:54+00:00,AE,un,photo,0,1,0,0,NaN,...,0,0,820,33,0,3,1,0,0.045122,2026-10-03 09:07:41+00:00
99986,7682349811001085191,2026-09-06 09:21:03+00:00,AE,un,video,13000,0,0,0,NaN,...,0,0,75,4,0,0,0,0,0.053333,2026-09-10 03:09:36+00:00
99992,7682349895788793106,2026-09-06 09:21:22+00:00,AE,un,video,16000,0,0,0,NaN,...,0,0,84,6,0,0,0,0,0.071429,2026-09-10 03:09:36+00:00
99995,7682349933487213842,2026-09-06 09:21:28+00:00,AE,un,video,15091,0,1080,1920,NaN,...,0,0,450,129,13,0,5,0,0.326667,2026-10-03 09:07:41+00:00


In [23]:

from lingua import LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_languages().build()

def detect(text):
    results = detector.compute_language_confidence_values(text)
    if not results:
        return 'un'
    best = results[0]                                    # the most likely language
    if best.value < 0.5:                                 # not sure enough
        return 'un'
    return best.language.iso_code_639_1.name.lower()     # e.g. ENGLISH -> 'en'

mask = (df['language'] == 'un') & df['caption'].notna()
df.loc[mask, 'language'] = df.loc[mask, 'caption'].apply(detect)

(df['language'] == 'un').sum()

np.int64(50773)

In [24]:
#NOTE ONLY AROUND 7K ENTRIES GOT FILLED


In [25]:
#frequency encoding for the data 
freq = df['language'].value_counts(normalize=True)   # each language -> its share of rows
df['lang_freq'] = df['language'].map(freq)           # replace each row's language with that share
df = df.drop(columns=['language'])

In [26]:
df.groupby('not_recommended_for_fyp')['is_ai_generated'].mean()

not_recommended_for_fyp
0    0.065107
1    0.098485
Name: is_ai_generated, dtype: float64

In [27]:
df['branded_content'].unique()

df['branded_content'].value_counts()

branded_content
                    99305
paid_partnership      686
shop_affiliate          9
Name: count, dtype: int64

In [28]:
df['is_branded'] = ((df['branded_content'] == 'paid_partnership') |
                    (df['branded_content'] == 'shop_affiliate')).astype(int)

df = df.drop(columns=['branded_content'])

In [29]:
df['sound_id'].unique() #analysis done on the whole dataset its better to keep this if we were training on the whole dataset but since we took only a part of it for this month 
#and its mostly 1.5 IDs for each row which gives us very little signal we drop for the full dataset we can since the ids repeate more 

df.drop(columns = 'sound_id', inplace = True)

KeyError: 'sound_id'

In [30]:
df['n_hashtags'] = df['hashtags'].apply(lambda x: len(x) if x is not None else 0)
df = df.drop(columns=['hashtag_ids'])

In [35]:
freq = df['country'].value_counts(normalize=True)
freq
df['country_freq'] = df['country'].map(freq)
df.drop(columns='country', inplace = True, errors= 'ignore')